<a href="https://colab.research.google.com/github/salmaelyagoubi/traitement_pdf/blob/main/traitement_pdf_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === 2. Configuration des dossiers Drive ===
SOURCE_FOLDER = "/content/drive/MyDrive/LOGISTIQUES/PDFs_A_traite/"
DEST_FOLDER = "/content/drive/MyDrive/LOGISTIQUES/PDFs_Modifies/"


# === 3. Installer les librairies nécessaires (une fois) ===
!pip install pdfplumber PyPDF2 reportlab

# === 4. Le traitement PDF (ton script adapté en fonction) ===
import re
from datetime import datetime
import pdfplumber
from PyPDF2 import PdfReader, PdfWriter
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
import io
import os

def process_pdf(input_path, output_path, threshold_hours=16):
    with pdfplumber.open(input_path) as pdf:
        lines = []
        for page in pdf.pages:
            if page.extract_text():
                lines.extend(page.extract_text().splitlines())

    first_start = None
    for line in lines:
        if "Heure de début de mission" in line:
            match = re.search(r'\b(\d{1,2})[:h](\d{2})\b', line)
            if match:
                first_start = datetime.strptime(f"{match[1]}:{match[2]}", "%H:%M")
                break

    if not first_start:
        print(f"❌ Pas d'heure de début dans {input_path}")
        return

    all_times = []
    for line in lines:
        if "retour" in line.lower() or "16/02" in line:
            continue
        found = re.findall(r'\b(\d{1,2})[:h](\d{2})\b', line)
        for h, m in found:
            all_times.append(datetime.strptime(f"{h}:{m}", "%H:%M"))

    valid_times = [t for t in all_times]
    last_time = max(valid_times) if valid_times else first_start

    duration = last_time - first_start
    total_minutes = duration.total_seconds() / 60
    hours = int(total_minutes // 60)
    minutes = int(total_minutes % 60)

    start_str = first_start.strftime("%H:%M")
    end_str = last_time.strftime("%H:%M")
    duration_str = f"{hours}h{minutes:02d}"

    message = (
        f"Heure de début : {start_str}\n"
        f"Heure de fin : {end_str}\n"
        f"Durée totale de la mission : {duration_str}\n"
    )
    if hours >= threshold_hours:
        message += "La personne a travaillé exceptionnellement longtemps aujourd’hui. Merci pour son implication."
    else:
        message += "La personne a accompli sa mission avec implication."

    # === Ajout du message dans le PDF ===
    packet = io.BytesIO()
    can = canvas.Canvas(packet, pagesize=A4)
    can.setFont("Helvetica-Bold", 11)
    y = 100
    for line in message.split("\n"):
        can.drawString(50, y, line)
        y += 15
    can.save()
    packet.seek(0)

    original = PdfReader(input_path)
    overlay = PdfReader(packet)
    writer = PdfWriter()

    for i, page in enumerate(original.pages):
        if i == len(original.pages) - 1:
            page.merge_page(overlay.pages[0])
        writer.add_page(page)

    with open(output_path, "wb") as f:
        writer.write(f)

    print(f"✅ {os.path.basename(input_path)} traité → {os.path.basename(output_path)}")

# === 5. Exécution du traitement ===
for file in os.listdir(SOURCE_FOLDER):
    if file.endswith(".pdf"):
        input_path = os.path.join(SOURCE_FOLDER, file)
        output_path = os.path.join(DEST_FOLDER, file.replace(".pdf", "_modifie.pdf"))
        process_pdf(input_path, output_path)
